# Data Quality

## Setup

In [290]:
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.patches import Patch

# Find the repo root by walking up until the warehouse folder is found,
# so the notebook works whatever folder the kernel starts in.
here = Path.cwd().resolve()
repo_root = next(p for p in [here, *here.parents] if (p / "warehouse" / "snap-assessment.duckdb").exists())

# read_only=True so the notebook never blocks `dbt build` from writing to the file.
con = duckdb.connect(str(repo_root / "warehouse" / "snap-assessment.duckdb"), read_only=True)

def q(sql: str) -> pd.DataFrame:
    """Run a SQL query and return the results as a Pandas DataFrame."""
    return con.sql(sql).df()

In [291]:
def df_profile(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame with descriptive statistics and missing-value metrics."""

    # 1. Standard description summary
    # For object columns, the result’s index includes count, unique, top, and freq. 
    # The top is the most common value and freq is its count. 
    # If multiple values tie for the highest count, top is chosen arbitrarily from among them.
    description = df.describe(include='all')

    # 2. Calculate NULL and blank metrics
    null_counts = df.isnull().sum()

    blank_counts = df.apply(
        lambda col: col.astype("string").str.strip().eq("").sum()
    )

    missing_counts = null_counts + blank_counts

    missing_percentages = (missing_counts / len(df) * 100).round(2)

    # 3. Create metrics DataFrame
    missing_info = pd.DataFrame(
        [
            null_counts,
            blank_counts,
            missing_counts,
            missing_percentages
        ],
        index=[
            'null_count',
            'blank_count',
            'missing_count',
            'missing_percent'
        ],
        columns=df.columns
    )

    # 4. Combine into comprehensive summary
    extended_description = pd.concat([description, missing_info])

    return extended_description

***Notes***

From Instructions: 
>The addresses in all of these datasets are synthetic addresses, but they are consistent between files for matching


- This noted because you will notice zip codes that do not match the provided city & state. I'm assuming made up matches for assessment.

## Per Table

### `card_list`

Merchant list derived from card activity, intended for prospecting.


In [292]:
df_card_list = q("select * exclude (_loaded_at) from raw.card_list")
df_profile(df_card_list)

,merchant_outlet_number,dba_name,addr1,city,state,zip_code,mcc_id,last_activity_period
count,1318,1318,1318,1318,1311,1318,1318,1318
unique,1318,1217,1025,650,49,960,32,7
top,31863874,AAMCO TRANSMISSIONS & TOTAL CAR CARE,2751 MORGAN RD,HOUSTON,DE,86909,7538,Txn within 13 months
freq,1,13,11,35,204,11,550,1047
null_count,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0
blank_count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
missing_count,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0
missing_percent,0.0,0.0,0.0,0.0,0.53,0.0,0.0,0.0


`merchant_outlet_number` is unique as expected, but business name and address look to potentially have some duplicates.

First a couple NULLs for the state field. Check out the rows.

In [293]:
## query to look for null state values in the card_list table
q("""

select
	* exclude (_loaded_at)
from "snap-assessment".raw.card_list
where state is null

""")

,merchant_outlet_number,dba_name,addr1,city,state,zip_code,mcc_id,last_activity_period
0,86998570,BOOST MOBILE,8274 RHODE ISLAND AVE NE,WASHINGTON,None,79981,4812,Txn within 13 months
1,89116306,TIRE 2 GO,661 CALLE VILLA,PONCE,None,99269,5532,Txn within 13 months
2,36690019,TIRE 2 GO DISTRIBUTORS #01761,661 CALLE VILLA,PONCE,None,992695388,7538,Txn between 14-24 months ago
3,31442926,TIRE 2 GO DISTRIBUTORS,661 CALLE VILLA,PONCE,None,992695388,5532,Txn within 13 months
4,21375975,TIRE 2 GO DISTRIBUTORS QUICKLU,661 CALLE VILLA,PONCE,None,992695388,7538,Txn within 13 months
5,12438351,TIRE 2 GO DISTRIBUTORS #04658,661 CALLE VILLA,PONCE,None,992695388,7538,Txn within 13 months
6,07790340,US-MOBILE1,8274 RHODE ISLAND AVE NE,WASHINGTON,None,79981,4812,Txn within 13 months


In [294]:
q("""

select
	* exclude (_loaded_at)
from "snap-assessment".raw.card_list
where lower(dba_name) ilike '%tire 2 go%'

""")

,merchant_outlet_number,dba_name,addr1,city,state,zip_code,mcc_id,last_activity_period
0,89116306,TIRE 2 GO,661 CALLE VILLA,PONCE,None,99269,5532,Txn within 13 months
1,36690019,TIRE 2 GO DISTRIBUTORS #01761,661 CALLE VILLA,PONCE,None,992695388,7538,Txn between 14-24 months ago
2,31442926,TIRE 2 GO DISTRIBUTORS,661 CALLE VILLA,PONCE,None,992695388,5532,Txn within 13 months
3,21375975,TIRE 2 GO DISTRIBUTORS QUICKLU,661 CALLE VILLA,PONCE,None,992695388,7538,Txn within 13 months
4,12438351,TIRE 2 GO DISTRIBUTORS #04658,661 CALLE VILLA,PONCE,None,992695388,7538,Txn within 13 months


Taking a glance at the dataset

In [295]:
df_card_list.head(10)

,merchant_outlet_number,dba_name,addr1,city,state,zip_code,mcc_id,last_activity_period
0,31863874,VELASQUEZ PERFORMANCE EXHAUST,8539 DOUGLAS RD,MONTGOMERY,PA,394618349,7538,Txn within 13 months
1,43801452,SOLIS TIRE CENTER,4276 AIRLINE DR,HOUSTON,DE,229235068,5532,Txn within 13 months
2,42439987,ROOMS TO GO 1701,88459 E US HIGHWAY 07,SEFFNER,TN,664152653,5712,Txn within 13 months
3,8575757,BROCCOLO AUTOMOTIVE,86 WELLSVILLE ST,ROCHESTER,ME,85376,7538,Txn within 13 months
4,15741126,GREAT DEALS TRUCK PARTS I,87381 S MAIN ST,LOS ANGELES,VA,099387672,5533,Txn within 13 months
5,44582271,FOREVERARTISANS.COM,4949 W ASHLAND WAY,FRANKLIN,AK,468671822,5094,Txn before 3 years ago
6,16766653,SPRINGS TIRE & AUTO LTD.,177 W COLORADO AVE,COLORADO SPRINGS,VT,19094,5532,No Txn
7,88294962,SAM'S APPLIANCE,82368 GRAND RIVER AVE,DETROIT,NH,51772,5722,Txn between 14-24 months ago
8,33948564,SOLID MOTORS LLC,778 S MCCLINTOCK DR,TEMPE,WV,147117788,7538,Txn within 13 months
9,31498360,CORNY`S SALES & SERVICE; INC.,3154 ROUTE 791,KNOX,IL,837674072,7538,Txn before 3 years ago


Some notes about the data when quickly glancing:
- all caps for name, address, and city
- zip code looks like it's Contains 5-digit and ZIP+4 formats without hypen

The `merchant_outlet_number` is unique, but is all the fields in the row unique?

In [296]:
q("""

select
    lower(trim(dba_name)) as company_name_clean,
    lower(trim(addr1)) as street_address_clean,
    lower(trim(city)) as city_clean,
    lower(trim(state)) as state_clean,
    case
        when REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
            then LEFT(TRIM(zip_code), 5)
        else NULL
    end as zip_code_5_digit, 
    -- mcc_id,
    count(*) as n_rows
from "snap-assessment".raw.card_list
group by all
having count(*) > 1 --If actually unique, this should return 0 rows
order by n_rows desc

""")

,company_name_clean,street_address_clean,city_clean,state_clean,zip_code_5_digit,n_rows
0,christian brothers automotive,3498 old denton rd,fort worth,de,23868,2
1,joes tires service,269 s santa fe st,el paso,de,20098,2
2,detroit wearmaster llc,1154 w 1 mile rd,detroit,nh,51778,2
3,igors auto body inc,834 commercial st,lynn,nj,98094,2
4,king of jewelers,7478 e hallandale beach blvd,hallandale beach,tn,66990,2
5,el che garage,892 w freeman st,duncanville,de,24883,2
6,wl appliances orlando,4940 edgewater dr,orlando,tn,67189,2
7,southwest delco wholesale distribution; l.l.c.,4098 spur 672,lubbock,de,20575,2
8,buesos furniture & more,687 n front st,warsaw,la,71601,2
9,big o tires,59579 california oaks rd,murrieta,va,07437,2


No, not all unique records even though unique IDs. 11 same (with some simple cleaning done) name, street address, city, state, zip code happen in more than one row... let's look at these rows individually.

In [297]:
q("""

with duplicates as (

select
    lower(trim(dba_name)) as company_name_clean,
    lower(trim(addr1)) as street_address_clean,
    lower(trim(city)) as city_clean,
    lower(trim(state)) as state_clean,
    case
        when REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
            then LEFT(TRIM(zip_code), 5)
        else NULL
    end as zip_code_5_digit, 
    count(*) as n_rows
from "snap-assessment".raw.card_list
group by all
having count(*) > 1 --If actually unique, this should return 0 rows

)
select
    merchant_outlet_number,
    dba_name as company_name_raw,
    lower(trim(dba_name)) as company_name_clean,
    addr1 as street_address_raw,
    lower(trim(addr1)) as street_address_clean,
    city as city_raw,
    lower(trim(city)) as city_clean,
    state as state_raw,
    lower(trim(state)) as state_clean,
    zip_code as zip_code_raw,
    case
        when REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
            then LEFT(TRIM(zip_code), 5)
        else NULL
    end as zip_code_5_digit,  
    case
        when REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{5}-[0-9]{4}$')
            then TRIM(zip_code)
        when REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{9}$')
            then LEFT(TRIM(zip_code), 5) || '-' || RIGHT(TRIM(zip_code), 4)
        else NULL
    end as zip_code_plus_4,
    mcc_id as mcc_id_list,
    last_activity_period
from "snap-assessment".raw.card_list r
join duplicates dupes
    on lower(trim(r.dba_name)) = dupes.company_name_clean
        and lower(trim(r.addr1)) = dupes.street_address_clean
        and lower(trim(r.city)) = dupes.city_clean
        and lower(trim(r.state)) = dupes.state_clean
order by company_name_clean, merchant_outlet_number

""")

,merchant_outlet_number,company_name_raw,company_name_clean,street_address_raw,street_address_clean,city_raw,city_clean,state_raw,state_clean,zip_code_raw,zip_code_5_digit,zip_code_plus_4,mcc_id_list,last_activity_period
0,27851993,BIG O TIRES,big o tires,59579 CALIFORNIA OAKS RD,59579 california oaks rd,MURRIETA,murrieta,VA,va,074374171,07437,07437-4171,7538,Txn before 3 years ago
1,62454962,BIG O TIRES,big o tires,59579 CALIFORNIA OAKS RD,59579 california oaks rd,MURRIETA,murrieta,VA,va,07437,07437,NaN,7538,Txn within 13 months
2,13161694,BUESOS FURNITURE & MORE,buesos furniture & more,687 N FRONT ST,687 n front st,WARSAW,warsaw,LA,la,716018649,71601,71601-8649,5021,Txn between 14-24 months ago
3,13166076,BUESOS FURNITURE & MORE,buesos furniture & more,687 N FRONT ST,687 n front st,WARSAW,warsaw,LA,la,71601,71601,NaN,5021,Txn between 14-24 months ago
4,06068854,CHRISTIAN BROTHERS AUTOMOTIVE,christian brothers automotive,3498 OLD DENTON RD,3498 old denton rd,FORT WORTH,fort worth,DE,de,23868,23868,NaN,7538,Txn between 14-24 months ago
5,43220579,CHRISTIAN BROTHERS AUTOMOTIVE,christian brothers automotive,3498 OLD DENTON RD,3498 old denton rd,FORT WORTH,fort worth,DE,de,23868,23868,NaN,7538,Txn before 3 years ago
6,07183148,DETROIT WEARMASTER LLC,detroit wearmaster llc,1154 W 1 MILE RD,1154 w 1 mile rd,DETROIT,detroit,NH,nh,51778,51778,NaN,7538,Txn within 13 months
7,35788872,DETROIT WEARMASTER LLC,detroit wearmaster llc,1154 W 1 MILE RD,1154 w 1 mile rd,DETROIT,detroit,NH,nh,517788801,51778,51778-8801,7538,Txn within 13 months
8,10495029,EL CHE GARAGE,el che garage,892 W FREEMAN ST,892 w freeman st,DUNCANVILLE,duncanville,DE,de,24883,24883,NaN,7538,Txn within 13 months
9,25353518,EL CHE GARAGE,el che garage,892 W FREEMAN ST,892 w freeman st,DUNCANVILLE,duncanville,DE,de,248835298,24883,24883-5298,7538,Txn within 13 months


Differences between the duplicates appear to have differences in one or more of these:
- zip code: 5 digit or 9 digit provided
- last_activity_period value
- mcc_id value (only for one: joes tires service)

### `purchased_list`

Third-party purchased business list with business classification and location enrichment attributes.


In [298]:
df_purchased_list = q("select * exclude (_loaded_at) from raw.purchased_list")
df_profile(df_purchased_list)

,speedeonid,name,alternative_name,street,city,state,postal_code,primary_sic_code_id,labels_primary_sic_code_id,location_employee_count,labels_square_footage
count,996,996,47,996,996,995,996,996,996,695,884
unique,996,906,19,897,616,49,899,32,35,39,8
top,421463432,Intoxalock Ignition Interlock,Intoxalock,1496 Whittier Blvd,Houston,VA,07876-8191,753801,Automobile Repairing & Service,2,"1 - 1,499"
freq,1,39,17,4,24,154,3,335,335,117,347
null_count,0.0,0.0,949.0,0.0,0.0,1.0,0.0,0.0,0.0,301.0,112.0
blank_count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
missing_count,0.0,0.0,949.0,0.0,0.0,1.0,0.0,0.0,0.0,301.0,112.0
missing_percent,0.0,0.0,95.28,0.0,0.0,0.1,0.0,0.0,0.0,30.22,11.24


`speedeonid` is an unique id as expected, but need to look into business name if same name also a different address.
The fields that would provide some enriched value about the account (`location_employee_count` and `labels_square_footage`) do have a decent amount of NULLs for something that is probably being paid for.

In [299]:
df_purchased_list.head(10)

,speedeonid,name,alternative_name,street,city,state,postal_code,primary_sic_code_id,labels_primary_sic_code_id,location_employee_count,labels_square_footage
0,421463432,T-Mart,NaN,6862 E Seminary Dr,Fort Worth,DE,23880-4368,571216,Furniture-Dealers-Retail,30,"40,000 - 99,999"
1,179674405,Redline Auto Works Performance,NaN,8627 Norwich Rd,Plainfield,UT,93625-8068,753801,Automobile Repairing & Service,NaN,NaN
2,573972274,Flores Tire SVC,NaN,73627 US 40 Rd,El Campo,DE,22562-4258,553111,Automobile Parts & Supplies-Retail-New,4,"2,500 - 4,999"
3,846053248,Fisher Furniture,NaN,183 Fairground Ave,Higginsville,MT,35962-8347,571220,Mattresses,2,"1,500 - 2,499"
4,777198756,Tuffy Tire & Auto Service,NaN,40 W Acorn Ln,Lake in the Hills,PA,39843-5195,553123,Tire-Dealers-Retail,6,"1 - 1,499"
5,630733922,Billy Bob's Repair & Tire,NaN,85791 Interstate 62 S,Elmendorf,DE,21887-0327,553117,Wheels & Wheel Covers,25,"20,000 - 39,999"
6,261502949,Ally Auto Service,NaN,1551 W Center Rd,Omaha,MS,31875-6862,753801,Automobile Repairing & Service,3,"1 - 1,499"
7,179437738,Intoxalock Ignition Interlock,NaN,6969 Osgood Ct,Fremont,VA,05460-4388,753834,Ignition Interlock Devices-Install & Service,NaN,NaN
8,689712314,Tire Rack,NaN,2898 Vorden Pkwy,South Bend,OR,53371-1577,553123,Tire-Dealers-Retail,300,"1 - 1,499"
9,172796311,Car Doctors Auto Repair,Turner Automotive Group Bda Hopewell Tire & Au...,6499 Jefferson Rd,Athens,SD,69392-8518,753801,Automobile Repairing & Service,5,"5,000 - 9,999"


In [300]:
q("""

select
    street,
    count(*) as n_rows,
    count(lower(trim("name"))) as n_biz_names,
    count(distinct postal_code) as n_zip_codes,
    count(distinct state) as n_state,
    count(distinct lower(city)) as n_cities
from "snap-assessment".raw.purchased_list
group by all
having count(*) > 1 
order by n_zip_codes desc, n_biz_names desc

""")

,street,n_rows,n_biz_names,n_zip_codes,n_state,n_cities
0,1496 Whittier Blvd,4,4,2,1,1
1,5599 Firestone Blvd,2,2,2,1,1
2,89698 Woodcrest Dr NW,3,3,1,1,1
3,84254 Monterey St,3,3,1,1,1
4,0669 N Whitaker Rd,3,3,1,1,1
...,...,...,...,...,...,...
83,5448 US 35,2,2,1,1,1
84,607 W Hidalgo Ave,2,2,1,1,1
85,7453 Bell Rd,2,2,1,1,1
86,72 Austin St,2,2,1,1,1


In [301]:
q("""

with dupe_street as (

select
    lower(trim(street)) as street,
    count(*) as n_rows,
    count(lower(trim("name"))) as n_biz_names,
    count(distinct trim(postal_code)) as n_zip_codes,
    count(distinct trim(state)) as n_state,
    count(distinct lower(trim(city))) as n_cities
from "snap-assessment".raw.purchased_list
group by all
having count(*) > 1 

)
select
    speedeonid,
    "name" as company_name_raw,
    lower(trim("name")) as company_name_clean,
    alternative_name as alternative_name_raw,
    lower(trim(alternative_name)) as alternative_name_clean, 	
    r.street as street_address_raw,
    lower(trim(r.street)) as street_address_clean,
    city as city_raw,
    lower(trim(city)) as city_clean,
    state as state_raw,
    lower(trim(state)) as state_clean,
    postal_code as zip_code_raw,
    CASE
    WHEN REGEXP_MATCHES(TRIM(postal_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
        THEN LEFT(TRIM(postal_code), 5)
    ELSE NULL
    END AS zip_code_5_digit,  
    CASE
        WHEN REGEXP_MATCHES(TRIM(postal_code), '^[0-9]{5}-[0-9]{4}$')
            THEN TRIM(postal_code)
        WHEN REGEXP_MATCHES(TRIM(postal_code), '^[0-9]{9}$')
            THEN LEFT(TRIM(postal_code), 5) || '-' || RIGHT(TRIM(postal_code), 4)
        ELSE NULL
    END AS zip_code_plus_4,
    primary_sic_code_id,
    labels_primary_sic_code_id,
    location_employee_count::int as location_employee_count,
    labels_square_footage
from "snap-assessment".raw.purchased_list r
join dupe_street dupes
    on lower(trim(r.street)) = dupes.street
order by street_address_clean, speedeonid

""")


,speedeonid,company_name_raw,company_name_clean,alternative_name_raw,alternative_name_clean,street_address_raw,street_address_clean,city_raw,city_clean,state_raw,state_clean,zip_code_raw,zip_code_5_digit,zip_code_plus_4,primary_sic_code_id,labels_primary_sic_code_id,location_employee_count,labels_square_footage
0,257410750,"Kaju Bros, Inc","kaju bros, inc",NaN,NaN,0243 Aero Dr,0243 aero dr,San Diego,san diego,VA,va,07876-8191,07876,07876-8191,753801,Automobile Repairing & Service,<NA>,"10,000 - 19,999"
1,296348196,Aero Auto Repair,aero auto repair,NaN,NaN,0243 Aero Dr,0243 aero dr,San Diego,san diego,VA,va,07876-8191,07876,07876-8191,753801,Automobile Repairing & Service,<NA>,"1 - 1,499"
2,597120512,Aero Auto Repair,aero auto repair,NaN,NaN,0243 Aero Dr,0243 aero dr,San Diego,san diego,VA,va,07876-8191,07876,07876-8191,753801,Automobile Repairing & Service,12,"2,500 - 4,999"
3,195063162,FleetPride Service Centers,fleetpride service centers,NaN,NaN,0361 McWhorter Rd,0361 mcwhorter rd,Dansville,dansville,ME,me,85562-0893,85562,85562-0893,753812,Truck-Repairing & Service,<NA>,NaN
4,525257326,"Bill Carpenter Service, Inc","bill carpenter service, inc",NaN,NaN,0361 McWhorter Rd,0361 mcwhorter rd,Dansville,dansville,ME,me,85562-0893,85562,85562-0893,753812,Truck-Repairing & Service,5,"1,500 - 2,499"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182,326752996,HOM Furniture,hom furniture,NaN,NaN,89698 Woodcrest Dr NW,89698 woodcrest dr nw,Coon Rapids,coon rapids,NV,nv,44566-3480,44566,44566-3480,571216,Furniture-Dealers-Retail,<NA>,"1,500 - 2,499"
183,546385486,HOM Furniture,hom furniture,NaN,NaN,89698 Woodcrest Dr NW,89698 woodcrest dr nw,Coon Rapids,coon rapids,NV,nv,44566-3480,44566,44566-3480,571216,Furniture-Dealers-Retail,300,"100,000+"
184,595013466,HOM Sleep Express,hom sleep express,NaN,NaN,89698 Woodcrest Dr NW,89698 woodcrest dr nw,Coon Rapids,coon rapids,NV,nv,44566-3480,44566,44566-3480,571220,Mattresses,800,"2,500 - 4,999"
185,211502987,Intoxalock Ignition Interlock,intoxalock ignition interlock,NaN,NaN,8974 N Central Ave,8974 n central ave,Medford,medford,IN,in,02498-4215,02498,02498-4215,753834,Ignition Interlock Devices-Install & Svc,<NA>,NaN


I'm assuming where the business name is actually different that the businesses are located in one big building, which does happen.

In [302]:
q("""

select
    lower(trim("name")) as business_name_clean,
    count(*) as n_rows,
    count(distinct postal_code) as n_zip_codes,
    count(distinct state) as n_state,
    count(distinct lower(city)) as n_cities,
    count(distinct lower(trim(street))) as n_streets
from "snap-assessment".raw.purchased_list
group by all
having count(*) > 1
order by n_zip_codes desc, n_rows desc

""")

,business_name_clean,n_rows,n_zip_codes,n_state,n_cities,n_streets
0,intoxalock ignition interlock,39,39,23,38,39
1,aamco,12,12,10,12,12
2,jiffy lube,9,9,6,9,9
3,midas,9,9,9,9,9
4,big o tires,7,7,5,7,7
5,discount tire,3,3,3,3,3
6,quick lane,3,3,3,3,3
7,christian brothers automotive,3,3,2,3,3
8,meineke,3,3,3,3,3
9,aero auto repair,3,2,1,1,2


If name on more than one row, the address (combo of zip, state, city, and street) is different which explains the multiple rows. The `aero auto repair` and `hom furniture` look to have potential dupes.

In [303]:
q("""

select
    * exclude (_loaded_at)
from "snap-assessment".raw.purchased_list r
where 1=1
    and lower(trim("name")) in ('aero auto repair', 'hom furniture')
order by name, speedeonid

""")

,speedeonid,name,alternative_name,street,city,state,postal_code,primary_sic_code_id,labels_primary_sic_code_id,location_employee_count,labels_square_footage
0,296348196,Aero Auto Repair,None,0243 Aero Dr,San Diego,VA,07876-8191,753801,Automobile Repairing & Service,NaN,"1 - 1,499"
1,597120512,Aero Auto Repair,None,0243 Aero Dr,San Diego,VA,07876-8191,753801,Automobile Repairing & Service,12,"2,500 - 4,999"
2,818597784,Aero Auto Repair,None,1134 Lake Murray Blvd,San Diego,VA,07880-7896,753801,Automobile Repairing & Service,NaN,"1 - 1,499"
3,326752996,HOM Furniture,None,89698 Woodcrest Dr NW,Coon Rapids,NV,44566-3480,571216,Furniture-Dealers-Retail,NaN,"1,500 - 2,499"
4,546385486,HOM Furniture,None,89698 Woodcrest Dr NW,Coon Rapids,NV,44566-3480,571216,Furniture-Dealers-Retail,300,"100,000+"


Definitely some duplicates. The question one, of the dupes which one is the correct to use to use the additional data about the company?

In [304]:
q("""

select
	lower(trim("name")) as prim_bus_name,
	alternative_name,
  count(*) as n_rows,
  count(distinct speedeonid) as n_speedeonid,
  count(distinct primary_sic_code_id) as n_primary_sic_code_ids
from "snap-assessment".raw.purchased_list
group by all
having count(*) > 1
order by n_primary_sic_code_ids desc, n_rows desc

""")

,prim_bus_name,alternative_name,n_rows,n_speedeonid,n_primary_sic_code_ids
0,bosch auto service,NaN,2,2,2
1,intoxalock ignition interlock,NaN,22,22,1
2,intoxalock ignition interlock,Intoxalock,17,17,1
3,aamco,AAMCO Transmissions & Total Car Care,12,12,1
4,jiffy lube,NaN,9,9,1
5,midas,NaN,9,9,1
6,big o tires,NaN,7,7,1
7,christian brothers automotive,NaN,3,3,1
8,quick lane,NaN,3,3,1
9,meineke,NaN,3,3,1


The business name "Bosch Auto Service" has two locations and each have a different primary sic code id. That field indicates the industry, so a bit odd the same business would be in different industries. However, this is the only case in this dataset.

### `salesforce_list`

CRM extract of lead records with status, vertical, opportunity sizing, and competitive information.

In [305]:
df_salesforce_list = q("""

select 
    id,
    company,
    street,
    city,
    state,
    zip5__c,
    industry_vertical__c,
    total_snap_opportunity_tso__c::float as total_snap_opportunity_tso__c,
    status,
    disqualified_reason__c,
    age_of_lead__c::int as age_of_lead__c,
    competitor_1_pll__c
from raw.salesforce_list

""")
df_profile(df_salesforce_list)

,id,company,street,city,state,zip5__c,industry_vertical__c,total_snap_opportunity_tso__c,status,disqualified_reason__c,age_of_lead__c,competitor_1_pll__c
count,1335,1335,1303,1272,1324,1271,1230,1.130000e+02,1335,631,1335.000000,312
unique,1335,1227,1162,672,47,831,15,NaN,10,15,NaN,15
top,99QUR99999W1KNJ7A6,Bob's Discount Furniture,571 Tolland Tpke,Houston,DE,99999,AUTO_SERVICE,NaN,Disqualified,Denied by Decisioning,NaN,No Tertiary
freq,1,7,7,34,224,24,444,NaN,650,146,NaN,110
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.382348e+05,NaN,NaN,736.489139,NaN
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.162195e+05,NaN,NaN,378.338355,NaN
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-4.236851e+01,NaN,NaN,0.000000,NaN
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.741149e+04,NaN,NaN,522.500000,NaN
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.759621e+04,NaN,NaN,728.000000,NaN
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.499427e+05,NaN,NaN,920.500000,NaN


The `id` is unique, but appears potential dupes for company+address. Some of the fields that are highly NULL are only filled when other field has a certian value, so the high missing percent is more expected.

In [306]:
df_salesforce_list.head(10)

,id,company,street,city,state,zip5__c,industry_vertical__c,total_snap_opportunity_tso__c,status,disqualified_reason__c,age_of_lead__c,competitor_1_pll__c
0,99QUR99999W1KNJ7A6,Richies Auto Service,772 South Belt Line Road,Irving,DE,24939,AUTO_SERVICE,29856.082031,Converted,NaN,140,No Tertiary
1,99Q4b999987pKb7EAE,To The Top Tires & Service,897 E Green St.,Macclesfield,LA,72147,TIRES_RIMS,NaN,Disqualified,QuikTrak,932,NaN
2,99QVQ99999cSCoL7AW,Wades Automotive Service Center,1669 Gratiot Ave,Detroit,NH,51786,AUTO_SERVICE,NaN,Working,NaN,7,NaN
3,99Q3g99999GeHM8EAN,Mee Consumables,NaN,NaN,MT,NaN,CELL_PHONES,NaN,Disqualified,Not an approved vertical,1805,NaN
4,99Q4b99999vhpnKEAQ,Great American home store,2669georgeln,NaN,NE,61362,OTHER,NaN,Disqualified,Not an approved vertical,1263,NaN
5,99Q4b999986yKBJEA7,Mattress Near Me Sacramento,4850 Madison Avenue,Sacramento,VA,04158,FURNITURE,NaN,Converted,NaN,805,Synchrony
6,99Q4b999984iHqlEAE,Gulf Appliance,5798 37nd Ave N Ste 87,Pinellas Park,TN,66218,APPLIANCES,NaN,Disqualified,Not an approved vertical,736,NaN
7,99Q3g99999IVjfIEAT,Wide Open Custom,8891 3th Street,Oswego,OH,NaN,OTHER,NaN,Disqualified,Not a Good Fit,1784,NaN
8,99QUR99999BNcmD7AT,B & L jewelry llc,"4191 Singleton Rd, Norcross GA 69906",Norcross,SD,69906,JEWELRY,NaN,Disqualified,Denied by Decisioning,577,NaN
9,99QVQ99999dNFxa7AG,West Virginia Glassworks,"867 N Main St, Moorefield WV 73163",Moorefield,AZ,73163,FURNITURE,NaN,Disqualified,Merchant already Exists in Snap App,1,NaN


- street
    - sometimes has the whole address and appears when it does the state in the address doesn't match the one in the state field
        - example:


| company | street | city | state | zip5__c |
|---|---|---|---|---|
| B & L jewelry llc | 4191 Singleton Rd, Norcross GA 69906 | Norcross | SD | 69906 |
| West Virginia Glassworks | 867 N Main St, Moorefield WV 7316 | Moorefield | AZ | 73163 |


Also, I noticed the zip code doesn't match either state in most cases, but need to remember:

>The addresses in all of these datasets are synthetic addresses, but they are consistent between files for matching

In [307]:
q(""" 

select
  count(*) as n_full_address,
  count(*) * 100 / (select count(*) from raw.website_list) as pct
from "snap-assessment".raw.salesforce_list
where street like '%,%'

""")

,n_full_address,pct
0,270,23.336214


~23% have the full address in the street field. Maybe should clean the street address by removing that City, State, and Zip. Just not sure what to think about the diff States.

In [308]:
q("""

select
  lower(trim(company)) as business_name_clean,
	count(*) as n_rows,
  count(distinct zip5__c) as n_zip_codes,
  count(distinct state) as n_state,
  count(distinct lower(city)) as n_cities,
  count(distinct lower(trim(street))) as n_streets
from "snap-assessment".raw.salesforce_list
group by all
having count(*) > 1
order by n_rows desc, n_zip_codes desc

""")

,business_name_clean,n_rows,n_zip_codes,n_state,n_cities,n_streets
0,bob's discount furniture,7,1,1,1,1
1,eyesouth partners,6,0,1,1,1
2,jerome's furniture,5,0,1,1,2
3,united automotive service,4,1,1,1,2
4,aj's transmisisons,4,1,1,1,2
...,...,...,...,...,...,...
106,newegg,2,0,1,1,1
107,ecs tuning,2,0,1,1,1
108,icebox diamonds & watches,2,0,1,1,1
109,american furniture warehouse,2,0,1,1,1


In [309]:
q("""

with raw_source as (

select
    id,
    company as company_name_raw,
    lower(trim(company)) as company_name_clean,
    street as street_address_raw,
    lower(trim(street)) as street_address_clean,
    city as city_raw,
    lower(trim(city)) as city_clean,
    state as state_raw,
    lower(trim(state)) as state_clean,
    zip5__c as zip_code_raw,
    CASE
        WHEN REGEXP_MATCHES(TRIM(zip5__c), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
            THEN LEFT(TRIM(zip5__c), 5)
        ELSE NULL
    END AS zip_code_5_digit, 
    industry_vertical__c,
    total_snap_opportunity_tso__c::float as total_snap_opportunity_tso__c,
    status,
    disqualified_reason__c,
    age_of_lead__c::int as age_of_lead__c,
    competitor_1_pll__c
from "snap-assessment".raw.salesforce_list

)
, duplicates as (

select
    company_name_clean,
    count(*) as n_rows,
    count(distinct zip_code_5_digit) as n_zip_codes,
    count(distinct state_clean) as n_state,
    count(distinct city_clean) as n_cities,
    count(distinct street_address_clean) as n_street
from raw_source
group by all
having n_rows > 1

)
, rank_dupes as (

select
    rank() over (partition by r.company_name_clean order by age_of_lead__c, status!='Disqualified' desc, total_snap_opportunity_tso__c is not null desc, zip_code_raw is not null desc, industry_vertical__c is not null desc, id desc) as dupe_record_rank,
    r.*
from raw_source r
join duplicates dupes
    on r.company_name_clean = dupes.company_name_clean
        and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows
        
)
select 
  *
from rank_dupes
order by company_name_clean, dupe_record_rank

""")

,dupe_record_rank,id,company_name_raw,company_name_clean,street_address_raw,street_address_clean,city_raw,city_clean,state_raw,state_clean,zip_code_raw,zip_code_5_digit,industry_vertical__c,total_snap_opportunity_tso__c,status,disqualified_reason__c,age_of_lead__c,competitor_1_pll__c
0,1,99QUR99999KMV8e7AH,501 Auto Repair,501 auto repair,"6334 Estes Pkwy, Longview TX 24396","6334 estes pkwy, longview tx 24396",Longview,longview,DE,de,24396,24396,AUTO_SERVICE,149942.734375,Disqualified,Denied by Decisioning,378,NaN
1,2,99QUR99999Hm8xi7AB,501 Auto Repair,501 auto repair,6334 Estes Pkwy,6334 estes pkwy,Longview,longview,DE,de,24396,24396,AUTO_SERVICE,NaN,Disqualified,Duplicate Lead,441,NaN
2,3,99QUR99999AXikT7AT,501 Auto Repair,501 auto repair,"6334 Estes Pkwy, Longview TX 24396","6334 estes pkwy, longview tx 24396",Longview,longview,DE,de,24396,24396,AUTO_SERVICE,NaN,Disqualified,Duplicate Lead,595,NaN
3,1,99Q4b999987r5jUEAQ,541 Furniture Gallery,541 furniture gallery,8399 N Riverside Ave,8399 n riverside ave,Medford,medford,IN,in,NaN,NaN,FURNITURE,NaN,Converted,NaN,423,NaN
4,2,99Q4b99999oxkIbEAI,541 Furniture Gallery,541 furniture gallery,8399 N Riverside Ave.,8399 n riverside ave.,Medford,medford,IN,in,02498,02498,FURNITURE,NaN,Disqualified,Duplicate Lead,1610,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,2,99Q4b999985RtaAEAS,WilsonBates Appliance Furniture Burley,wilsonbates appliance furniture burley,7439 Overland Ave,7439 overland ave,Burley,burley,RI,ri,16681,16681,FURNITURE,NaN,Converted,NaN,832,NaN
238,1,99QUR99999ZEwb37AD,xtreme autosports scv,xtreme autosports scv,79284 Soledad Cyn Rd.,79284 soledad cyn rd.,Canyon Country,canyon country,VA,va,08648,08648,TIRES_RIMS,NaN,Disqualified,Duplicate Lead,76,NaN
239,2,99QUR99999ZEub57AD,xtreme autosports scv,xtreme autosports scv,79284 soledad canyon road,79284 soledad canyon road,Canyon Country,canyon country,VA,va,08648,08648,TIRES_RIMS,NaN,Disqualified,Duplicate Lead,76,NaN
240,1,99QUR99999GxfNh7AJ,Zoes JC Furniture,zoes jc furniture,4391 Pinemont Drive,4391 pinemont drive,Houston,houston,DE,de,22907,22907,FURNITURE,625063.687500,Converted,NaN,387,Progressive


### `snap`

Internal merchant list with current status and year-to-date production metrics. Please note that all KPI's here are meant to mirror snap trends, but are not actual values associated with any snap locations.



In [310]:
df_snap = q("""

select
    mid,
    dba_name,
    industry,
    product_type,
    merchant_address,
    merchant_city,
    merchant_state,
    merchant_zip_code, 
    is_inactivated::boolean as is_inactivated,
    apps_ytd::int as apps_ytd,
    aprv_ytd::int as aprv_ytd,
    cmpl_ytd::int as cmpl_ytd,
    fund_ytd::numeric as fund_ytd
from raw.snap

""")
df_profile(df_snap)

,mid,dba_name,industry,product_type,merchant_address,merchant_city,merchant_state,merchant_zip_code,is_inactivated,apps_ytd,aprv_ytd,cmpl_ytd,fund_ytd
count,1222,1222,1222,1222,1222,1222,1222,1222,1222,573.0,573.0,573.0,4.830000e+02
unique,1222,1205,15,2,988,574,45,808,2,<NA>,<NA>,<NA>,NaN
top,16814,Solis Tire Center,AUTO_SERVICE,Lease,5948 816rd St,Houston,VA,07437,False,<NA>,<NA>,<NA>,NaN
freq,1,2,441,829,5,55,275,7,654,<NA>,<NA>,<NA>,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.287958,43.232112,28.722513,5.095603e+04
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,156.776953,82.105234,53.404494,9.431644e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,6.740000e+02
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,6.0,3.0,7.392500e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.0,17.0,10.0,2.033100e+04
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,74.0,43.0,30.0,5.424250e+04


The `mid` is unique, but appears potential dupes for dba_name+address. Some of the fields that are highly NULL are only filled when `is_inactivated=false`, so high percent is valid.

In [311]:
df_snap.head(10)

,mid,dba_name,industry,product_type,merchant_address,merchant_city,merchant_state,merchant_zip_code,is_inactivated,apps_ytd,aprv_ytd,cmpl_ytd,fund_ytd
0,16814,C Jaes Automotive Repair,AUTO_SERVICE,Loan,15 N 8799 W,Orem,CT,15942,False,8,8,5,5258.0
1,50830,Vehicle Vibes,CAR_AUDIO,Lease,87 Enterprise Rd Ste 6,Hyannis,NJ,97398,False,1,0,0,NaN
2,17289,Audio Experience Tulsa,CAR_AUDIO,Lease,4098 S 892th East,Tulsa,IA,25883,False,60,34,21,26546.0
3,79048,"A-warehouse Furniture, Llc",FURNITURE,Lease,8081 W Cedar Bayou Lynchburg Rd,Baytown,DE,22478,False,192,149,95,154359.0
4,08053,Gills Point S Tire & Auto Service 5120 - Albuq...,TIRES_RIMS,Lease,4198 5th St Nw,Albuquerque,MD,12892,False,121,102,71,60304.0
5,75430,Mike's Westside Auto,AUTO_SERVICE,Lease,7758 Porter Rd,Katy,DE,22506,False,88,79,50,81085.0
6,00773,Aamco Transmissions-glendale,AUTO_SERVICE,Loan,8555 E Colorado St,Glendale,VA,08794,False,1,0,0,NaN
7,22930,Emmanuel Appliances,APPLIANCES,Lease,8467 Broad St,Cranston,ID,97094,False,188,104,60,155795.0
8,886731,El Paisa Tire And Wheels Shop Armitage,TIRES_RIMS,Lease,"5182 W Armitage Ave, Chicago Il 39360",Chicago,PA,39360,False,11,6,10,3683.0
9,886720,Maximum Offroad Lv,AUTO_SERVICE,Loan,6424 W Naples Dr,Las Vegas,MN,10896,False,7,2,5,1595.0


In [312]:
q("""

select
  lower(trim(dba_name)) as company_name_clean,
  count(*) as n_rows,
  count(distinct merchant_zip_code) as n_zip_codes,
  count(distinct merchant_state) as n_state,
  count(distinct lower(trim(merchant_city))) as n_cities,
  count(distinct lower(trim(merchant_address))) as n_street
from "snap-assessment".raw.snap
group by all
order by n_rows desc, n_zip_codes desc

""")

,company_name_clean,n_rows,n_zip_codes,n_state,n_cities,n_street
0,solis tire center,2,2,1,1,2
1,aero auto repair,2,1,1,1,1
2,american auto care and tire,2,1,1,1,1
3,economy tire & wheel,2,1,1,1,1
4,price busters - marlboro pike,2,1,1,1,1
...,...,...,...,...,...,...
1200,paws and performance,1,1,1,1,1
1201,sunrise auto center,1,1,1,1,1
1202,arman jewelers lw,1,1,1,1,1
1203,kwik kar east allen,1,1,1,1,1


The n_rows > 1 and n_zip_codes + n_state + n_cities + n_street = 4 indicate duplicates. Let's look at a more granular level.

In [313]:
q("""

with raw_source as (

select
    mid,
    dba_name as company_name_raw,
    lower(trim(dba_name)) as company_name_clean,
    industry,
    product_type,
    merchant_address as street_address_raw,
    lower(trim(merchant_address)) as street_address_clean,
    merchant_city as city_raw,
    lower(trim(merchant_city)) as city_clean,
    merchant_state as state_raw,
    lower(trim(merchant_state)) as state_clean,
    merchant_zip_code as zip_code_raw,
    CASE
    WHEN REGEXP_MATCHES(TRIM(merchant_zip_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
        THEN LEFT(TRIM(merchant_zip_code), 5)
    ELSE NULL
    END AS zip_code_5_digit, 
    is_inactivated::boolean as is_inactivated,
    apps_ytd::int as apps_ytd,
    aprv_ytd::int as aprv_ytd,
    cmpl_ytd::int as cmpl_ytd,
    fund_ytd::numeric as fund_ytd
from "snap-assessment".raw.snap

)
, duplicates as (

select
  lower(trim(dba_name)) as company_name_clean,
  count(*) as n_rows,
  count(distinct merchant_zip_code) as n_zip_codes,
  count(distinct merchant_state) as n_state,
  count(distinct lower(trim(merchant_city))) as n_cities,
  count(distinct lower(trim(merchant_address))) as n_street
from "snap-assessment".raw.snap
group by all
having n_rows > 1

)
, rank_dupes as (

select
    r.*,
    rank() over (partition by r.company_name_clean order by is_inactivated, mid::int desc) as record_rank
from raw_source r
join duplicates dupes
    on r.company_name_clean = dupes.company_name_clean
        and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows

)
select 
  *
from rank_dupes
order by company_name_clean, mid

""")

,mid,company_name_raw,company_name_clean,industry,product_type,street_address_raw,street_address_clean,city_raw,city_clean,state_raw,state_clean,zip_code_raw,zip_code_5_digit,is_inactivated,apps_ytd,aprv_ytd,cmpl_ytd,fund_ytd,record_rank
0,64414,41st Street Automotive Sioux Falls,41st street automotive sioux falls,AUTO_SERVICE,Lease,090 W 58st St,090 w 58st st,Sioux Falls,sioux falls,GA,ga,42894,42894,True,<NA>,<NA>,<NA>,NaN,2
1,71272,41st Street Automotive Sioux Falls,41st street automotive sioux falls,AUTO_SERVICE,Lease,090 W 58st St,090 w 58st st,Sioux Falls,sioux falls,GA,ga,42894,42894,True,<NA>,<NA>,<NA>,NaN,1
2,60339,Aamco Pantego,aamco pantego,AUTO_SERVICE,Loan,7679 W Pioneer Pkwy,7679 w pioneer pkwy,Pantego,pantego,DE,de,23986,23986,True,<NA>,<NA>,<NA>,NaN,2
3,60341,Aamco Pantego,aamco pantego,AUTO_SERVICE,Loan,7679 W Pioneer Pkwy,7679 w pioneer pkwy,Pantego,pantego,DE,de,23986,23986,True,<NA>,<NA>,<NA>,NaN,1
4,23730,Aero Auto Repair,aero auto repair,AUTO_SERVICE,Loan,0243 Aero Dr,0243 aero dr,San Diego,san diego,VA,va,07876,07876,True,<NA>,<NA>,<NA>,NaN,2
5,56770,Aero Auto Repair,aero auto repair,AUTO_SERVICE,Loan,0243 Aero Dr,0243 aero dr,San Diego,san diego,VA,va,07876,07876,False,119,74,60,116506.0,1
6,69178,American Auto Care And Tire,american auto care and tire,AUTO_SERVICE,Loan,7859 Marconi Ave,7859 marconi ave,Sacramento,sacramento,VA,va,04178,04178,True,<NA>,<NA>,<NA>,NaN,2
7,84123,American Auto Care And Tire,american auto care and tire,AUTO_SERVICE,Loan,7859 Marconi Ave,7859 marconi ave,Sacramento,sacramento,VA,va,04178,04178,True,<NA>,<NA>,<NA>,NaN,1
8,61063,Anytime Auto,anytime auto,AUTO_SERVICE,Loan,779 Enterprise Rd NE,779 enterprise rd ne,Rio Rancho,rio rancho,MD,md,12875,12875,True,<NA>,<NA>,<NA>,NaN,2
9,61116,Anytime Auto,anytime auto,AUTO_SERVICE,Loan,779 Enterprise Rd NE,779 enterprise rd ne,Rio Rancho,rio rancho,MD,md,12875,12875,True,<NA>,<NA>,<NA>,NaN,1


Some business have exact values for all fields expect the `mid`, so plan to dedupe by picking the larger mid assuming the one added more recently. Not always the best to pick the most recent, but with current circumstances of time and lack of knowledge of complete data this is the assumption I will be making for now. Some had a difference in the `is_inactivated` field, in those cases the ones where the field equals false is kept.

### `website_list`

Lead intake data captured from website forms, including self-reported business and vertical details.


In [314]:
df_website_list = q("select * exclude (_loaded_at) from raw.website_list")
df_profile(df_website_list)

,form_name,lead_id,business_name,street_address,city,state,zip_code,industry_name,industry_desc,industry_subcategory,industry_subcategory_desc
count,1157,1157,1157,1049,1049,1046,1049,966,966,966,966
unique,2,1103,943,847,578,47,702,4,4,29,29
top,LONG_FORM,456b9fa0-0903-5962-0412-6de65ef52f2a,SACRAMENTO AUTO SOLUTIONS,"7899 Fulton Ave, Sacramento CA 04174",Houston,DE,04174,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
freq,1049,5,8,11,45,233,13,747,747,453,453
null_count,0.0,0.0,0.0,108.0,108.0,111.0,108.0,191.0,191.0,191.0,191.0
blank_count,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
missing_count,0.0,0.0,0.0,108.0,108.0,111.0,108.0,191.0,191.0,191.0,191.0
missing_percent,0.0,0.0,0.0,9.33,9.33,9.59,9.33,16.51,16.51,16.51,16.51


No unique id in this data set.

In [315]:
df_website_list.head(10)

,form_name,lead_id,business_name,street_address,city,state,zip_code,industry_name,industry_desc,industry_subcategory,industry_subcategory_desc
0,LONG_FORM,296fe82e-af5f-55e2-03aa-003473e0124e,Mynors Auto Repair,8835 e baseline st,San Bernardino,VA,07589,AUTOMOTIVE,AUTOMOTIVE,COLLISION_REPAIR,Auto Collision & Repair
1,LONG_FORM,24925c6e-6004-529b-024a-699877f7df51,J & J Car Care,"3982 Harford Rd, Baltimore MD 78785",Baltimore,NM,78785,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
2,LONG_FORM,09a461e1-f9f1-5922-ae4e-4e7d357c739e,ADVANCE MUFFLER AND AUTO REPAIR,"1154 W 1 Mile Rd, Detroit MI 51778",Detroit,NH,51778,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
3,LONG_FORM,186b8d01-deba-5d70-1d17-ac4c05263d44,Power Lube Express Auto Care,"104 S Broadway St, Blackfoot ID 16778",Blackfoot,RI,16778,AUTOMOTIVE,AUTOMOTIVE,TIRES_RIMS,Tires & Rims
4,LONG_FORM,ae8d5fd4-4247-54b5-b181-534baba1a42b,PURRFECT AUTO SERVICE 99,"1496 Whittier Blvd Ste B8, Pico Rivera CA 09339",Pico Rivera,VA,09339,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
5,LONG_FORM,22ecca3f-5d52-5fa3-bd2f-3f210261eaf1,Lee Motorsports,"7282 Gum Rd, Seneca MO 35134",Seneca,MT,35134,AUTOMOTIVE,AUTOMOTIVE,MOTORCYCLE_ATV_UTV_PARTS_SERVICE,Motorcycle/ATV/UTV Parts & Service
6,LONG_FORM,78c1345e-22a3-507a-a419-2e207cec9a32,Eureka auto tech,"8754 Eureka Rd, Wyandotte MI 51807",Wyandotte,NH,51807,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
7,LONG_FORM,1be57672-822d-5dde-b388-a8d7476a7f36,a shade darker window tinting and accessories,"7942 Highway 48 S, Covington TN 61980",Covington,FL,61980,AUTOMOTIVE,AUTOMOTIVE,CAR_AUDIO,Car Audio
8,LONG_FORM,4864b15d-5ad8-5103-bcf9-b65bbdd6981a,My Mechanic,"7497 Onslow Dr, Jacksonville NC 71459",Jacksonville,LA,71459,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service
9,LONG_FORM,87bd94dd-6b83-5434-1d22-ee1a62ab49b1,Byers Tire & Auto,"184 W Denver St, Greenwood AR 27063",Greenwood,WA,27063,AUTOMOTIVE,AUTOMOTIVE,TIRES_RIMS,Tires & Rims


- street_address
    - sometimes has the whole address and appears when it does the state in the address doesn't match the one in the state field
        - example:

| business_name | street_address | city | state | zip_code |
|---|---|---|---|---|
| Mynors Auto Repair | 8835 e baseline st | San Bernardino | VA | 07589 |
| J & J Car Care | 3982 Harford Rd, Baltimore MD 78785 | Baltimore | NM | 78785 |
| ADVANCE MUFFLER AND AUTO REPAIR | 1154 W 1 Mile Rd, Detroit MI 51778 | Detroit | NH | 51778 |
| Power Lube Express Auto Care | 104 S Broadway St, Blackfoot ID 16778 | Blackfoot | RI | 16778 |
| PURRFECT AUTO SERVICE 99 | 1496 Whittier Blvd Ste B8, Pico Rivera CA 09339 | Pico Rivera | VA | 09339 |

- also noticed the zip code doesn't match either state in most cases, but need to remember: 
>The addresses in all of these datasets are synthetic addresses, but they are consistent between files for matching

- in other settings, this would make me feel the web form that the user is filling out is not working as expected

In [316]:
q("""

select
  count(*) as n_full_address,
  count(*) * 100 / (select count(*) from raw.website_list) as pct
from "snap-assessment".raw.website_list
where street_address like '%,%'

""")

,n_full_address,pct
0,615,53.15471


A decent percent. Maybe should clean the street address by removing that City, State, and Zip. Just not sure what to think about the diff States.

In [317]:
q("""

with raw_source as (

select
    form_name,
    lead_id,
    business_name as company_name_raw,
    lower(trim(business_name)) as company_name_clean,
    street_address as street_address_raw,
    lower(trim(street_address)) as street_address_clean,
    city as city_raw,
    lower(trim(city)) as city_clean,
    state as state_raw,
    lower(trim(state)) as state_clean,
    zip_code as zip_code_raw,
    CASE
        WHEN REGEXP_MATCHES(TRIM(zip_code), '^[0-9]{5}(-[0-9]{4}|[0-9]{4})?$')
            THEN LEFT(TRIM(zip_code), 5)
        ELSE NULL
    END AS zip_code_5_digit,  
    industry_name,
    industry_desc,
    industry_subcategory,
    industry_subcategory_desc
from "snap-assessment".raw.website_list

)
, duplicates as (

select
  company_name_clean,
  count(*) as n_rows,
  count(distinct zip_code_5_digit) as n_zip_codes,
  count(distinct state_clean) as n_state,
  count(distinct city_clean) as n_cities,
  count(distinct street_address_clean) as n_street
from raw_source
group by all
having n_rows > 1

)
, rank_dupes as (
select
    r.*,
    rank() over (partition by r.company_name_clean order by street_address_raw is not null desc, industry_name is not null desc, industry_subcategory!='OTHER' desc, lead_id desc, industry_subcategory) as record_rank
from raw_source r
join duplicates dupes
    on r.company_name_clean = dupes.company_name_clean
        and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows and dupes.n_zip_codes < dupes.n_rows
)
select 
  *
from rank_dupes
order by company_name_clean, record_rank

""")

,form_name,lead_id,company_name_raw,company_name_clean,street_address_raw,street_address_clean,city_raw,city_clean,state_raw,state_clean,zip_code_raw,zip_code_5_digit,industry_name,industry_desc,industry_subcategory,industry_subcategory_desc,record_rank
0,LONG_FORM,ab3a030d-db2b-5694-b084-587847c323e4,465 truck and trailer repair inc,465 truck and trailer repair inc,"6879 S Arlington Ave, Indianapolis IN 53796","6879 s arlington ave, indianapolis in 53796",Indianapolis,indianapolis,OR,or,53796,53796,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,1
1,LONG_FORM,ab3a030d-db2b-5694-b084-587847c323e4,465 truck and trailer repair inc,465 truck and trailer repair inc,"6879 S Arlington Ave, Indianapolis IN 53796","6879 s arlington ave, indianapolis in 53796",Indianapolis,indianapolis,OR,or,53796,53796,AUTOMOTIVE,AUTOMOTIVE,COMMERCIAL_TIRES,Commercial Tires,2
2,LONG_FORM,d94449ae-a776-5395-0f85-785cb6ef45e7,501 Auto Repair,501 auto repair,"6334 Estes Pkwy, Longview TX 24396","6334 estes pkwy, longview tx 24396",Longview,longview,DE,de,24396,24396,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,1
3,LONG_FORM,5fb8d79f-35cd-5541-ae3e-845c44d4a25f,501 Auto Repair,501 auto repair,6334 Estes Pkwy,6334 estes pkwy,Longview,longview,DE,de,24396,24396,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,2
4,LONG_FORM,3c877577-6c95-5d66-adea-473a8536be76,501 Auto Repair,501 auto repair,"6334 Estes Pkwy, Longview TX 24396","6334 estes pkwy, longview tx 24396",Longview,longview,DE,de,24396,24396,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
391,LONG_FORM,d56393d1-c8b2-59b7-062d-ec5e93894ced,wades automotive service center,wades automotive service center,"1669 Gratiot Ave, Detroit MI 51786","1669 gratiot ave, detroit mi 51786",Detroit,detroit,NH,nh,51786,51786,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,1
392,SHORT_FORM,b0596b7c-cc70-5ec6-bc71-2947f6a6a0e4,wades automotive service center,wades automotive service center,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
393,SHORT_FORM,0d14369a-22a2-5ba8-00e8-b50ef90e9eed,wades automotive service center,wades automotive service center,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
394,LONG_FORM,56a02591-e1e1-5b9a-03b8-4361de28c56e,Wayne County Tire and Service,wayne county tire and service,7360 Savannah Hwy,7360 savannah hwy,Jesup,jesup,SD,sd,68454,68454,AUTOMOTIVE,AUTOMOTIVE,AUTO_SERVICE,Auto Service,1


**short form** doesn't have anything beyond the company name. It might not be the most helpful data to use right away. I would wonder if something is wrong with the form. See below.

In [ ]:
q("""

select
  *
from "snap-assessment".raw.website_list
where form_name='SHORT_FORM'
limit 10

""")

ConnectionException: Connection Error: Connection already closed!

## Combining Data

This section just to note that there is no common ID between any of the tables to easily join. Then as pointed out above, company names and addresses aren't the cleanest and duplicate problems. Thus joining these tables will easily result in more messy data, missed joins as name is slightly diff, or over join if to lay back how the fields used to join.

## Summary

In [318]:
# Scorecard: one row per dataset, built from the df_profile output above

KEY_FIELDS = {   # (dataframe, name, street, zip, key attribute)
    "snap":            (df_snap,            "dba_name",      "merchant_address", "merchant_zip_code", "industry"),
    "card_list":       (df_card_list,       "dba_name",      "addr1",            "zip_code",          "mcc_id"),
    "purchased_list":  (df_purchased_list,  "name",          "street",           "postal_code",       "location_employee_count"),
    "salesforce_list": (df_salesforce_list, "company",       "street",           "zip5__c",           "total_snap_opportunity_tso__c"),
    "website_list":    (df_website_list,    "business_name", "street_address",   "zip_code",          "industry_subcategory"),
}

rows = []
for dataset, (df, name_c, street_c, zip_c, attr_c) in KEY_FIELDS.items():
    miss = df_profile(df).loc["missing_percent"]
    rows.append({
        "dataset": dataset,
        "n_rows": len(df),
        "name_missing_pct": float(miss[name_c]),
        "street_missing_pct": float(miss[street_c]),
        "zip_missing_pct": float(miss[zip_c]),
        "key_attr": attr_c,
        "key_attr_missing_pct": float(miss[attr_c]),
    })
scorecard = pd.DataFrame(rows)

# Duplicates: the rule lives in the staging models (dupe_record_rank), so cite that on the slide
dupes = q("""
select 'snap' as dataset, round(100.0 * count(*) filter (where dupe_record_rank > 1) / count(*), 1) as dupe_pct from "snap-assessment".staging.stg_snap
union all select 'card_list',       round(100.0 * count(*) filter (where dupe_record_rank > 1) / count(*), 1) from "snap-assessment".staging.stg_card_list
union all select 'purchased_list',  round(100.0 * count(*) filter (where dupe_record_rank > 1) / count(*), 1) from "snap-assessment".staging.stg_purchased_list
union all select 'salesforce_list', round(100.0 * count(*) filter (where dupe_record_rank > 1) / count(*), 1) from "snap-assessment".staging.stg_salesforce_list
union all select 'website_list',    round(100.0 * count(*) filter (where dupe_record_rank > 1) / count(*), 1) from "snap-assessment".staging.stg_website_list
""")
scorecard = scorecard.merge(dupes, on="dataset")

# Starting thresholds are my assumption. Adjust them and say so on the slide.
def rate(r):
    worst = max(r.name_missing_pct, r.street_missing_pct, r.zip_missing_pct)
    if r.dupe_pct < 2 and worst < 5:   return "High"
    if r.dupe_pct < 10 and worst < 20: return "Medium"
    return "Low"

scorecard["rating"] = scorecard.apply(rate, axis=1)
scorecard


,dataset,n_rows,name_missing_pct,street_missing_pct,zip_missing_pct,key_attr,key_attr_missing_pct,dupe_pct,rating
0,snap,1222,0.0,0.00,0.00,industry,0.00,1.3,High
1,card_list,1318,0.0,0.00,0.00,mcc_id,0.00,0.3,High
2,purchased_list,996,0.0,0.00,0.00,location_employee_count,30.22,0.2,High
3,salesforce_list,1335,0.0,2.40,4.79,total_snap_opportunity_tso__c,91.54,4.8,Medium
4,website_list,1157,0.0,9.33,9.33,industry_subcategory,16.51,12.6,Low


### So what: what I trust each dataset for

| Dataset | Trust it for | Be careful about |
|---|---|---|
| snap | What a good merchant looks like (industry, approval and completion rates, funded $) | KPIs are simulated, so I treat patterns as directional. 1.3% duplicate rows. |
| card_list | Freshness: `last_activity_period` shows who is still transacting | Addresses are all caps with mixed ZIP formats. 0.3% duplicates. MCC needs a lookup to be useful. |
| purchased_list | Cleanest addresses, plus size signals (employees, square footage) | 30.22% of employee counts are null, which is a lot for a paid file. Same business name can appear with different SIC codes. |
| salesforce_list | Pipeline status, TSO, competitor and disqualify reasons | 4.8% duplicate company+address rows. Street sometimes holds a full address. Status meaning (is `Converted` already in Snap?) needs confirming. |
| website_list | Inbound intent, the warmest leads | Messiest fields (typed by the lead). No lead ID uniqueness guarantee. 12.6% duplicates. |

**Assumptions I made**
- Addresses are synthetic, so city, state and ZIP often disagree. I match on name, street and ZIP, not state.
- For duplicates I keep one row per business. In Snap I keep the active one, then the higher MID, assuming it's newer.
- I kept Salesforce rows that aren't `Disqualified` when choosing among duplicates.
- Website and Salesforce rows with a full address in the street field, I plan to cut the city, state, and ZIP but ignoring that state doesn't match the value in the `state field`.
- A business is the same business if cleaned name, street and ZIP5 match exactly. Fuzzy matching is a next step.

**Overall:** the data is usable for prioritizing by segment and recency, but not yet for one-to-one lead matching. I would not give Sales a lead list without a match-quality check.


## close connection

In [320]:
con.close()